<a href="https://colab.research.google.com/github/Ramprasath-Meganathan/LoRA-Plus/blob/main/ML_Project_LoRA%2B.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Master toggles - Change to True ONLY when necessary
RUN_INSTALLATION = False
RUN_HEAVY_TRAINING = False
DOWNLOAD_DATASETS = False

In [2]:
#Save Libraries to Google Drive for ease of Installation
from google.colab import drive
import os
import sys

# Create a path for your persistent library storage
lib_path = '/content/drive/MyDrive/colab_env/'

if os.path.exists(lib_path):
  print("Google drive is already mounted and verified. Skipping Authentication...")
else:
  print("Target path not detected. Initializing active Google Drive mount pipeline...")
  drive.mount('/content/drive')

  #Enforce directory creation if its a completely fresh drive
  !mkdir -p {lib_path}
  print("Drive mounted and folder verified.")

# CLEANUP: Remove the slow, unzipped Drive directory from sys.path to avoid circular imports
if lib_path in sys.path:
  sys.path.remove(lib_path)
  print("Removed slow Google Drive path from sys.path to prevent import conflicts.")

Target path not detected. Initializing active Google Drive mount pipeline...
Mounted at /content/drive
Drive mounted and folder verified.


In [11]:
# Clean and robust installation of only the missing or outdated libraries
try:
    import peft
    import evaluate
    import torchao
    # Ensure torchao is upgraded to avoid peft compatibility issues
    from importlib.metadata import version
    assert int(version('torchao').split('.')[1]) >= 16
    print("Essential packages are already installed and meet version requirements!")
except (ImportError, AssertionError):
    print("Installing or upgrading missing packages 'peft', 'evaluate', and 'torchao' natively...")
    !pip install -q -U peft evaluate torchao
    print("Installation complete!")

Installing or upgrading missing packages 'peft', 'evaluate', and 'torchao' natively...
Installation complete!


In [10]:
# Legacy ZIP Loader disabled as libraries are now natively handled via standard fast-install

In [12]:
# Clean up any legacy search paths and clear Python's internal module cache to prevent contamination
import sys
legacy_path = "/content/local_hf_env"

# Remove from search path
while legacy_path in sys.path:
    sys.path.remove(legacy_path)

# Purge loaded modules originating from the old local staging directory from memory
modules_to_purge = []
for name, module in list(sys.modules.items()):
    if module and hasattr(module, '__file__') and module.__file__ and legacy_path in module.__file__:
        modules_to_purge.append(name)

for name in modules_to_purge:
    del sys.modules[name]

# Necessary Library Imports
import os
import math
import time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import transformers
import datasets
import peft
from datasets import load_dataset
from torch.optim import AdamW
from google.colab import drive
from peft import get_peft_model, LoraConfig, TaskType
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments
)

In [5]:
print("PyTorch", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)

PyTorch 2.11.0+cu130
Transformers: 5.22.0
Datasets: 4.8.5
PEFT: 0.21.2


In [6]:
# Verify GPU environment (Google Colab T4 16GB limit)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name: {torch.cuda.get_device_name(0)}")

CUDA Available: True
Device Name: Tesla T4


In [7]:
# Mount Google Drive for saving checkpoints to protect against session timeouts
drive.mount('/content/drive')
CHECKPOINT_DIR = "/content/drive/MyDrive/LoRA_Plus_Checkpoints/"
if os.path.exists('/content/drive/MyDrive'):
  print("Google drive is already verified. Skipping mount request")
else:
  print("Target mount point not detected. Mounting Google Drive...")
  drive.mount('/content/drive')

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print(f"Checkpoints will be saved to: {CHECKPOINT_DIR}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google drive is already verified. Skipping mount request
Checkpoints will be saved to: /content/drive/MyDrive/LoRA_Plus_Checkpoints/


In [13]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Load the dataset
dataset = load_dataset("nyu-mll/glue", "sst2")

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

def tokenize_function(batch):
  return tokenizer(
      batch["sentence"],
      truncation=True,
      max_length=128
  )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True)

tokenized_dataset = tokenized_dataset.remove_columns(
    ["sentence", "idx"]
)

tokenized_dataset = tokenized_dataset.rename_column(
    "label", "labels"
)

tokenized_dataset.set_format("torch")

In [14]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("roberta-base")

tokens = tokenizer.tokenize("The movie is amazing.")
print(tokens)

encoded = tokenizer("The movie is amazing.")
print(tokenizer.convert_ids_to_tokens(encoded["input_ids"]))

['The', 'Ġmovie', 'Ġis', 'Ġamazing', '.']
['<s>', 'The', 'Ġmovie', 'Ġis', 'Ġamazing', '.', '</s>']


In [15]:
print(tokenized_dataset)

DatasetDict({
    train: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['labels', 'input_ids', 'attention_mask'],
        num_rows: 1821
    })
})


### 🧪 Pilot Testing Configuration (1,000 Examples)
Run this cell to extract a small slice of your dataset to test your hyperparameters, GPU limits, and evaluation metrics before committing to the full 67k training set.

In [41]:
# Toggle this to False when you are ready for the full 67k run!
PILOT_RUN = False

if PILOT_RUN:
    # Subset the tokenized datasets safely
    pilot_train_dataset = tokenized_dataset["train"].select(range(1000))
    pilot_eval_dataset = tokenized_dataset["validation"].select(range(min(200, len(tokenized_dataset["validation"]))))
    print(f"⚡ Pilot Mode Active: Subsampled Train ({len(pilot_train_dataset)} examples) and Eval ({len(pilot_eval_dataset)} examples).")
else:
    pilot_train_dataset = tokenized_dataset["train"]
    pilot_eval_dataset = tokenized_dataset["validation"]
    print(f"🔥 Full Mode Active: Training on all {len(pilot_train_dataset)} examples.")

🔥 Full Mode Active: Training on all 67349 examples.


To utilize this subset, update your `trainer` block in cell `c3RtHY5yfQFH` to use the variables `train_dataset=pilot_train_dataset` and `eval_dataset=pilot_eval_dataset` instead of `tokenized_dataset["train"]` and `tokenized_dataset["validation"]` directly.

In [42]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=["query","value"],
    modules_to_save=["classifier"],
    bias="none"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066


In [33]:
from transformers import AutoModelForSequenceClassification
from peft import LoraConfig, get_peft_model, TaskType

model = AutoModelForSequenceClassification.from_pretrained(
    "roberta-base",
    num_labels=2
)

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=8,
    lora_alpha=8,
    lora_dropout=0.1,
    target_modules=["query","value"],
    modules_to_save=["classifier"],
    bias="none"
)

model = get_peft_model(model, lora_config)

model.print_trainable_parameters()


Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                        | Status     | 
---------------------------+------------+-
lm_head.layer_norm.weight  | UNEXPECTED | 
lm_head.dense.weight       | UNEXPECTED | 
lm_head.bias               | UNEXPECTED | 
lm_head.dense.bias         | UNEXPECTED | 
lm_head.layer_norm.bias    | UNEXPECTED | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


trainable params: 887,042 || all params: 125,534,212 || trainable%: 0.7066


In [34]:
import torch

trainable_parameters = []

for parameter in model.parameters():
    if parameter.requires_grad:
        trainable_parameters.append(parameter)

optimizer = torch.optim.AdamW(
    trainable_parameters,
    lr=1e-4,
    weight_decay=0.01
)

In [43]:
lora_a_parameters = []
lora_b_parameters = []
other_parameters = []

for name, parameter in model.named_parameters():
  if not parameter.requires_grad:
    continue

  if "lora_A" in name:
    lora_a_parameters.append(parameter)
  elif "lora_B" in name:
    lora_b_parameters.append(parameter)
  else:
    other_parameters.append(parameter)

base_lr = 1e-4
# lambda_ratio = 1
lambda_ratio = 4  # Scale up LoRA_B parameter learning rate by 8x for LoRA+

optimizer = torch.optim.AdamW(
    [
        {
            "params": lora_a_parameters,
            "lr": base_lr
        },
        {
            "params": lora_b_parameters,
            "lr": base_lr * lambda_ratio
        },
        {
            "params": other_parameters,
             "lr": base_lr
        }
    ],
    weight_decay=0.01
)

In [44]:
import torch
import evaluate
import numpy as np

from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction

    predictions = np.argmax(predictions, axis=-1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/LoRA_Plus_Project/loraplus_8x",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    logging_strategy="steps",
    logging_steps=50,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    num_train_epochs=3,

    fp16=torch.cuda.is_available(),

    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True
)

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=pilot_train_dataset,
    eval_dataset=pilot_eval_dataset,

    # train_dataset=tokenized_dataset["train"],
    # eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics,

    optimizers=(optimizer, None)
)

trainer.train()

Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [45]:
import torch
import evaluate
import numpy as np

from transformers import (
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

accuracy = evaluate.load("accuracy")

def compute_metrics(eval_prediction):
    predictions, labels = eval_prediction

    predictions = np.argmax(predictions, axis=-1)

    return accuracy.compute(
        predictions=predictions,
        references=labels
    )

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/LoRA_Plus_Project/loraplus_8x",

    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,

    logging_strategy="steps",
    logging_steps=50,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    num_train_epochs=3,

    fp16=torch.cuda.is_available(),

    report_to="none",

    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    greater_is_better=True
)

trainer = Trainer(
    model=model,
    args=training_args,

    train_dataset=pilot_train_dataset,
    eval_dataset=pilot_eval_dataset,

    # train_dataset=tokenized_dataset["train"],
    # eval_dataset=tokenized_dataset["validation"],

    processing_class=tokenizer,
    data_collator=data_collator,

    compute_metrics=compute_metrics,

    optimizers=(optimizer, None)
)

trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.231518,0.206285,0.936927
2,0.192042,0.233479,0.933486
3,0.211406,0.224739,0.935780


TrainOutput(global_step=12630, training_loss=0.22046957854137283, metrics={'train_runtime': 814.9319, 'train_samples_per_second': 247.931, 'train_steps_per_second': 15.498, 'total_flos': 3769070742192264.0, 'train_loss': 0.22046957854137283, 'epoch': 3.0})

In [37]:
# Print out the exact evaluation history to diagnose accuracy behavior
for log in trainer.state.log_history:
    if 'eval_accuracy' in log:
        print(f"Epoch {log.get('epoch', 'N/A')}: Eval Loss = {log.get('eval_loss', 'N/A'):.4f}, Eval Accuracy = {log.get('eval_accuracy', 'N/A'):.4f}")

Epoch 1.0: Eval Loss = 0.1974, Eval Accuracy = 0.9346
Epoch 2.0: Eval Loss = 0.2138, Eval Accuracy = 0.9381
Epoch 3.0: Eval Loss = 0.2074, Eval Accuracy = 0.9335


In [ ]:
# Force kill the VM runtime container right after execution finishes
from google.colab import runtime
runtime.unassign()